# Time Series Analysis - Homework 1
> **Author:** Elias Schweighuber  
> **Course:** TIS  
>
> **Tasks:**
> 1. **Load the data:** Load the CSV files with Polars and measure the loading time.
> 2. **Inspect and combine the sales files:**
>    - Inspect the `sales_train_*` and `sales_test_*` dataframes (4 files: `sales_train_validation`, `sales_test_validation`, `sales_train_evaluation`, `sales_test_evaluation`).
>    - Check which days each file covers so that no day is counted twice.
>    - Join the data into a single dataframe covering the entire timeline ($d_1$ to $d_{1969}$).
>    - Replace integer day columns ($d_1, d_2, \dots$) with real dates ($d_1$ = 29 January 2011).
>    - Persist the result as one Parquet file.

## 1. Load the Data
Load all CSV files with Polars and measure their respective loading times.

In [38]:
import time
from datetime import date, timedelta
from pathlib import Path
import polars as pl


In [39]:
DATA_DIR = Path('data')

csv_files = sorted(DATA_DIR.glob('*.csv'))
dfs = {}
load_times = {}


for filepath in csv_files:
    fname = filepath.name
    file_size_mb = filepath.stat().st_size / (1024 * 1024)
    
    t0 = time.perf_counter()
    df = pl.read_csv(filepath)
    elapsed = time.perf_counter() - t0
    
    dfs[fname] = df
    load_times[fname] = elapsed
    

# summary
summary_df = pl.DataFrame({
    'file': [f.name for f in csv_files],
    'rows': [dfs[f.name].height for f in csv_files],
    'columns': [dfs[f.name].width for f in csv_files],
    'size_mb': [round(f.stat().st_size / (1024 * 1024), 2) for f in csv_files],
    'load_time_sec': [round(load_times[f.name], 4) for f in csv_files]
})
summary_df

file,rows,columns,size_mb,load_time_sec
str,i64,i64,f64,f64
"""calendar.csv""",1969,13,0.11,0.0032
"""sales_test_evaluation.csv""",30490,33,3.11,0.0174
"""sales_test_validation.csv""",30490,33,2.82,0.0169
"""sales_train_evaluation.csv""",30490,1946,115.56,0.6998
"""sales_train_validation.csv""",30490,1918,113.91,0.843
"""sample_submission.csv""",60980,29,4.99,0.0258
"""sell_prices.csv""",6841121,4,226.09,0.8268
"""weights_evaluation.csv""",42840,5,2.15,0.0118
"""weights_validation.csv""",42840,5,2.14,0.0124


## 2. Inspect and Combine the Sales Files

### 2.1 Inspect the Sales DataFrames

In [40]:
sales_filenames = [
    'sales_train_validation.csv',
    'sales_test_validation.csv',
    'sales_train_evaluation.csv',
    'sales_test_evaluation.csv'
]

sales_inspection = []
for fname in sales_filenames:
    df = dfs[fname]
    index_cols = [c for c in df.columns if not c.startswith('d_')]
    day_cols = [c for c in df.columns if c.startswith('d_')]
    day_numbers = [int(c.split('_')[1]) for c in day_cols]
    
    min_day = min(day_numbers)
    max_day = max(day_numbers)
    
    sales_inspection.append({
        'file': fname,
        'rows': df.height,
        'total_cols': df.width,
        'index_cols': ', '.join(index_cols),
        'num_day_cols': len(day_cols),
        'day_range': f'd_{min_day} to d_{max_day}',
    })

pl.DataFrame(sales_inspection)

file,rows,total_cols,index_cols,num_day_cols,day_range
str,i64,i64,str,i64,str
"""sales_train_validation.csv""",30490,1918,"""item_id, dept_id, cat_id, stor…",1913,"""d_1 to d_1913"""
"""sales_test_validation.csv""",30490,33,"""item_id, dept_id, cat_id, stor…",28,"""d_1914 to d_1941"""
"""sales_train_evaluation.csv""",30490,1946,"""item_id, dept_id, cat_id, stor…",1941,"""d_1 to d_1941"""
"""sales_test_evaluation.csv""",30490,33,"""item_id, dept_id, cat_id, stor…",28,"""d_1942 to d_1969"""


**Analysis of the 4 Sales Files & Day Coverage:**


In [41]:
# Check overlap between sales_train_validation and sales_train_evaluation
val_d_cols = [c for c in dfs['sales_train_validation.csv'].columns if c.startswith('d_')]
val_overlap_match = all(
    (dfs['sales_train_validation.csv'][c] == dfs['sales_train_evaluation.csv'][c]).all()
    for c in val_d_cols
)
print(f"overlap d_1..d_1913 (train_validation vs train_evaluation): {'Identical' if val_overlap_match else 'Mismatch!'}")

# Check overlap between sales_test_validation and sales_train_evaluation
test_val_d_cols = [c for c in dfs['sales_test_validation.csv'].columns if c.startswith('d_')]
test_val_overlap_match = all(
    (dfs['sales_test_validation.csv'][c] == dfs['sales_train_evaluation.csv'][c]).all()
    for c in test_val_d_cols
)
print(f"Overlap d_1914..d_1941 (test_validation vs train_evaluation): {'Identical' if test_val_overlap_match else 'Mismatch!'}")

# Check sales_test_evaluation
test_eval_d_cols = [c for c in dfs['sales_test_evaluation.csv'].columns if c.startswith('d_')]
print(f"sales_test_evaluation adds 28 days: {test_eval_d_cols[0]} to {test_eval_d_cols[-1]}")

overlap d_1..d_1913 (train_validation vs train_evaluation): Identical
Overlap d_1914..d_1941 (test_validation vs train_evaluation): Identical
sales_test_evaluation adds 28 days: d_1942 to d_1969


### 2.2 Join the data into a single DataFrame
* We take `sales_train_evaluation` which already cleanly combines d_1 through d_1941.
* We join `sales_test_evaluation` which provides days d_1942 through d_1969

In [48]:
eval_df = dfs['sales_train_evaluation.csv']
test_eval_df = dfs['sales_test_evaluation.csv']
test_eval_day_cols = [c for c in test_eval_df.columns if c.startswith('d_')]

combined_df = eval_df.join(
    test_eval_df.select(index_cols + test_eval_day_cols),
    on=index_cols,
    how='inner'
)

# sort day columns
all_day_cols = [c for c in combined_df.columns if c.startswith('d_')]
sorted_day_cols = sorted(all_day_cols, key=lambda c: int(c.split('_')[1]))

# index columns followed by sorted day columns
final_col_order = index_cols + sorted_day_cols
combined_df = combined_df.select(final_col_order)

print(f"DataFrame shape: {combined_df.shape}")
print(f"Total day columns: {len(sorted_day_cols)} ({sorted_day_cols[0]} to {sorted_day_cols[-1]})")

DataFrame shape: (30490, 1974)
Total day columns: 1969 (d_1 to d_1969)


### 2.3 Replace Integer Day Columns with Real Dates
> Replace the integer day columns (d1, d2, ...) with real dates. Day d1 is 29 January 2011.

In [51]:
start_date = date(2011, 1, 29)

# date mapping dictionary
day_to_date_map = {
    col: (start_date + timedelta(days=int(col.split('_')[1]) - 1)).strftime('%Y-%m-%d')
    for col in sorted_day_cols
}

# Rename
sales_df_dated = combined_df.rename(day_to_date_map)

date_cols = [day_to_date_map[c] for c in sorted_day_cols]
print(f"First date column:  {date_cols[0]} (Day d_1)")
print(f"Last date column:   {date_cols[-1]} (Day d_1969)")

sales_df_dated.head(3)

First date column:  2011-01-29 (Day d_1)
Last date column:   2016-06-19 (Day d_1969)


item_id,dept_id,cat_id,store_id,state_id,2011-01-29,2011-01-30,2011-01-31,2011-02-01,2011-02-02,2011-02-03,2011-02-04,2011-02-05,2011-02-06,2011-02-07,2011-02-08,2011-02-09,2011-02-10,2011-02-11,2011-02-12,2011-02-13,2011-02-14,2011-02-15,2011-02-16,2011-02-17,2011-02-18,2011-02-19,2011-02-20,2011-02-21,2011-02-22,2011-02-23,2011-02-24,2011-02-25,2011-02-26,2011-02-27,2011-02-28,2011-03-01,…,2016-05-14,2016-05-15,2016-05-16,2016-05-17,2016-05-18,2016-05-19,2016-05-20,2016-05-21,2016-05-22,2016-05-23,2016-05-24,2016-05-25,2016-05-26,2016-05-27,2016-05-28,2016-05-29,2016-05-30,2016-05-31,2016-06-01,2016-06-02,2016-06-03,2016-06-04,2016-06-05,2016-06-06,2016-06-07,2016-06-08,2016-06-09,2016-06-10,2016-06-11,2016-06-12,2016-06-13,2016-06-14,2016-06-15,2016-06-16,2016-06-17,2016-06-18,2016-06-19
str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""HOBBIES_1_001""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,4,0,0,0,0,3,3,0,1,2,0,1,0,0,1,4,3,0,0,1,1,0,2,0,0,0,3,2,1,2,0,0,1,0,1,3,1
"""HOBBIES_1_002""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,2,1,1,0,0,0,0,0,0,2,0,1,0,1,0,1,0,0,0,1,1,1,1,0,0,0,1,0,0,1,0,0,2,1,1,0
"""HOBBIES_1_003""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,2,0,0,0,2,3,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,1,0,1,3,2,1,0,2,1,0,1,1


### 2.4 Persist the Result as One Parquet File
> Persist the result as one Parquet file.

In [56]:
output_parquet_path = DATA_DIR / 'sales_combined.parquet'

sales_df_dated.write_parquet(output_parquet_path, compression='zstd')

parquet_size_mb = output_parquet_path.stat().st_size / (1024 * 1024)
print(f"Parquet file size:      {parquet_size_mb:.2f} MB")

Parquet file size:      20.07 MB
